In [3]:

# JAX / GPU sanity check
import jax
import jax.numpy as jnp
import equinox as eqx
import optax

print("=== JAX ===")
print("version      :", jax.__version__)
print("devices      :", jax.devices())
print("default backend:", jax.default_backend())

# Basic compute check
key = jax.random.PRNGKey(0)
x = jax.random.normal(key, (1024, 1024))
y = jnp.dot(x, x.T)
print(f"\nMatrix multiply (1024×1024) result shape: {y.shape}, sum: {float(y.sum()):.2f}")

print("\n=== Equinox ===")
print("version:", eqx.__version__)

# Tiny 1-layer MLP smoke test
mlp = eqx.nn.MLP(in_size=8, out_size=4, width_size=16, depth=1, key=key)
z = mlp(jnp.ones(8))
print("MLP output:", z)

print("\n=== Optax ===")
print("version:", optax.__version__)
opt = optax.adamw(1e-3)
params = eqx.filter(mlp, eqx.is_array)
opt_state = opt.init(params)
print("OptState type:", type(opt_state))

print("\nAll checks passed ✓")


=== JAX ===
version      : 0.6.2
devices      : [CpuDevice(id=0)]
default backend: cpu

Matrix multiply (1024×1024) result shape: (1024, 1024), sum: 1011297.69

=== Equinox ===
version: 0.13.6
MLP output: [-0.26229656  0.07357499  0.14641792  0.06358334]

=== Optax ===
version: 0.2.8
OptState type: <class 'tuple'>

All checks passed ✓


In [2]:

# Verify project source packages are importable
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from src.data.loader    import load_raw
from src.data.dataset   import make_windows, numpy_dataloader
from src.models.transformer1d import DiffusionTransformer1D
from src.models.diffusion     import DiffusionProcess
from src.training.train       import Trainer
from src.evaluation.metrics   import run_all_metrics

print("All src.* imports OK ✓")

# Tiny forward pass through the denoiser
import jax, jax.numpy as jnp
model = DiffusionTransformer1D(
    seq_len=96, d_model=64, n_heads=4, n_layers=2, d_ff=128,
    n_clusters=3, n_day_types=2, key=jax.random.PRNGKey(1),
)
x_t = jax.random.normal(jax.random.PRNGKey(2), (96,))
t   = jnp.array(42)
c   = jnp.array([0, 1])
out = model(x_t, t, c)
print(f"Denoiser forward pass OK — output shape: {out.shape}, mean: {float(out.mean()):.4f}")


All src.* imports OK ✓
Denoiser forward pass OK — output shape: (96,), mean: -0.0894
